# 🔧 02B — Patch: Generate Labeled Pairs untuk SetFit (Opsi B)
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Apa yang dilakukan notebook ini?

SetFit membutuhkan **labeled pair data** — pasangan (teks_A, teks_B) dengan label:
- `1` = RELEVAN — judul skripsi cocok dengan profil dosen pembimbing aslinya
- `0` = TIDAK RELEVAN — judul skripsi dipasangkan dengan dosen yang bukan pembimbingnya

Data ini dibuat secara otomatis dari CSV skripsi yang sudah ada.

```
skripsi_clean.csv + profil_dosen_clean.csv
        ↓
  Positif: (judul_skripsi, profil_dosen_asli)         → label 1
  Negatif: (judul_skripsi, profil_dosen_lain_random)  → label 0
        ↓
  labeled_pairs.csv  → dipakai di 04B_setfit.ipynb
```

> ✅ **Notebook 02 (preprocessing) tidak perlu dijalankan ulang.**
> Notebook ini cukup dijalankan **satu kali** setelah 02_preprocessing selesai,
> dan **sebelum** menjalankan 04B_setfit.ipynb.

---
## 🔧 LANGKAH 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))

import config
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
print('✅ Setup selesai.')

---
## 📌 LANGKAH 1 — Load Data

In [ ]:
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_CLEAN)
df_dosen   = pd.read_csv(config.FILE_DOSEN_CLEAN)

df_skripsi['teks_bersih_bert'] = df_skripsi['teks_bersih_bert'].fillna('')
df_dosen['profil_bersih_bert'] = df_dosen['profil_bersih_bert'].fillna('')

# Hanya skripsi yang pembimbingnya ada di daftar dosen
dosen_valid = set(df_dosen['nama_dosen'].tolist())
df_skripsi  = df_skripsi[df_skripsi['pembimbing'].isin(dosen_valid)].reset_index(drop=True)

# Buat dict: nama_dosen → profil_teks
profil_map = dict(zip(df_dosen['nama_dosen'], df_dosen['profil_bersih_bert']))
nama_dosen_list = df_dosen['nama_dosen'].tolist()

print('✅ Data dimuat.')
print(f'   Skripsi valid  : {len(df_skripsi)}')
print(f'   Dosen          : {len(df_dosen)}')

---
## 📌 LANGKAH 2 — Generate Labeled Pairs

In [ ]:
# ─── KONFIGURASI ─────────────────────────────────────────────────
# Rasio negatif:positif — 2:1 adalah titik keseimbangan yang baik
# Terlalu banyak negatif → model bias menolak semua
# Terlalu sedikit negatif → model tidak belajar membedakan
NEG_PER_POS = 2

print(f'⚙️  Konfigurasi:')
print(f'   Pasangan positif (1 per skripsi)  : {len(df_skripsi)}')
print(f'   Pasangan negatif per skripsi      : {NEG_PER_POS}')
print(f'   Total pairs estimasi              : ~{len(df_skripsi) * (1 + NEG_PER_POS)}')

In [ ]:
# ─── GENERATE PAIRS ───────────────────────────────────────────────
pairs = []

for _, row in df_skripsi.iterrows():
    teks_skripsi  = row['teks_bersih_bert']
    dosen_asli    = row['pembimbing']
    profil_asli   = profil_map.get(dosen_asli, '')

    # ── POSITIF: pasangkan dengan dosen pembimbing asli ──
    pairs.append({
        'teks_skripsi'  : teks_skripsi,
        'teks_dosen'    : profil_asli,
        'nama_dosen'    : dosen_asli,
        'judul'         : row['judul'],
        'label'         : 1,
    })

    # ── NEGATIF: pasangkan dengan dosen lain secara random ──
    dosen_lain = [d for d in nama_dosen_list if d != dosen_asli]
    dosen_neg  = random.sample(dosen_lain, min(NEG_PER_POS, len(dosen_lain)))

    for dosen_n in dosen_neg:
        pairs.append({
            'teks_skripsi'  : teks_skripsi,
            'teks_dosen'    : profil_map.get(dosen_n, ''),
            'nama_dosen'    : dosen_n,
            'judul'         : row['judul'],
            'label'         : 0,
        })

df_pairs = pd.DataFrame(pairs)

# Acak urutan agar tidak ada pola berurutan positif-negatif-negatif
df_pairs = df_pairs.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'✅ Labeled pairs berhasil dibuat!')
print(f'   Total pairs   : {len(df_pairs)}')
print(f'   Positif (1)   : {(df_pairs["label"]==1).sum()}')
print(f'   Negatif (0)   : {(df_pairs["label"]==0).sum()}')
print(f'   Rasio neg:pos : {(df_pairs["label"]==0).sum() / (df_pairs["label"]==1).sum():.1f}:1')

In [ ]:
# ─── PREVIEW ─────────────────────────────────────────────────────
print('📋 5 contoh labeled pairs:\n')
for _, row in df_pairs.head(5).iterrows():
    label_str = '✅ RELEVAN' if row['label'] == 1 else '❌ TIDAK RELEVAN'
    print(f'  Label      : {label_str}')
    print(f'  Skripsi    : {row["judul"][:65]}...')
    print(f'  Dosen      : {row["nama_dosen"]}')
    print(f'  Teks dosen : {str(row["teks_dosen"])[:60]}...')
    print()

In [ ]:
# ─── VISUALISASI DISTRIBUSI PAIRS ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart label
label_counts = df_pairs['label'].value_counts()
axes[0].pie(label_counts, labels=['Tidak Relevan (0)', 'Relevan (1)'],
            colors=['#EF5350','#66BB6A'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Distribusi Label Pairs', fontweight='bold')

# Bar: jumlah pair per dosen (positif)
pos_pairs = df_pairs[df_pairs['label']==1]
dosen_counts = pos_pairs['nama_dosen'].value_counts()
nama_pendek = [n.split(',')[0].split('.')[-1].strip()[:15] for n in dosen_counts.index]
axes[1].bar(nama_pendek, dosen_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Jumlah Pasangan Positif per Dosen', fontweight='bold')
axes[1].set_xlabel('Dosen')
axes[1].set_ylabel('Jumlah Pair')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'labeled_pairs_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 📌 LANGKAH 3 — Simpan Labeled Pairs

In [ ]:
out_path = os.path.join(config.DATA_PROCESSED, 'labeled_pairs.csv')
df_pairs.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'💾 labeled_pairs.csv tersimpan: {out_path}')
print(f'   Jumlah baris : {len(df_pairs)}')
print(f'   Kolom        : {list(df_pairs.columns)}')
print()
print('✅ File ini siap digunakan di 04B_setfit.ipynb')

---
## ✅ Selesai — Ringkasan Notebook 02B

| File | Lokasi | Keterangan |
|------|--------|------------|
| `labeled_pairs.csv` | `data/processed/` | Input untuk SetFit training |
| `labeled_pairs_dist.png` | `results/` | Visualisasi distribusi pairs |

### ⚠️ Notebook Lain yang Tidak Perlu Diubah:
- `00_setup.ipynb` — tidak perlu diubah
- `01_load_data.ipynb` — tidak perlu diubah
- `02_preprocessing.ipynb` — tidak perlu diubah (**ini hanya tambahan**, bukan pengganti)
- `03_tfidf_baseline.ipynb` — tidak perlu diubah
- `04A_bert_embedding.ipynb` — tidak perlu diubah

### 🗺️ Langkah Berikutnya:
> Jalankan **`04B_setfit.ipynb`** untuk proses fine-tuning